# 동적 오케스트레이터와 워커

중앙 모델이 입력을 읽고 필요한 하위 작업의 개수와 지시를 정합니다.

## 실행 환경과 비용 상한

오케스트레이터 1회, 워커 최대 4회, 종합 1회로 제한합니다. 계획 비용은 0.0183달러 미만입니다.

In [1]:
# 이 장은 OpenAI SDK를 직접 배우므로 두 패키지만 고정합니다.
%pip install -q openai==2.29.0 python-dotenv==1.2.3


Note: you may need to restart the kernel to use updated packages.


In [2]:
# 비동기 API 호출과 사용량 측정에 필요한 도구를 준비합니다.
import asyncio
import json
import os
import re
import time
from collections import Counter
from dotenv import find_dotenv, load_dotenv
from openai import AsyncOpenAI
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "프로젝트 루트 .env에 OPENAI_API_KEY가 필요합니다."
MODEL = "gpt-5.4-mini"
client = AsyncOpenAI(timeout=45.0, max_retries=0)
calls: list[dict[str, float | int | str]] = []
print("모델:", MODEL, "| API 키 존재: 예")


모델: gpt-5.4-mini | API 키 존재: 예


In [3]:
# 비동기 Responses API의 JSON 응답과 사용량을 기록합니다.
async def call_json(instructions: str, prompt: str, limit: int = 220) -> dict:
    started = time.perf_counter()
    response = await client.responses.create(
        model=MODEL,
        instructions=instructions + " 반드시 유효한 JSON 객체만 반환하세요.",
        input="JSON 요청\n" + prompt,
        max_output_tokens=limit,
        reasoning={"effort": "none"},
        text={"format": {"type": "json_object"}},
    )
    calls.append({"latency_s": time.perf_counter() - started,
                  "input": response.usage.input_tokens,
                  "output": response.usage.output_tokens})
    return json.loads(response.output_text)


In [4]:
# 비동기 응답의 answer 형식을 엄격하게 검사합니다.
async def call_answer(instructions: str, prompt: str, limit: int = 400) -> dict:
    started = time.perf_counter()
    response = await client.responses.create(
        model=MODEL, instructions=instructions, input=prompt,
        max_output_tokens=limit, reasoning={"effort": "none"},
        text={"format": {"type": "json_schema", "name": "answer_payload",
              "schema": {"type": "object", "properties": {"answer": {"type": "string"}},
              "required": ["answer"], "additionalProperties": False}, "strict": True}},
    )
    calls.append({"latency_s": time.perf_counter() - started,
                  "input": response.usage.input_tokens, "output": response.usage.output_tokens})
    data = json.loads(response.output_text)
    if not isinstance(data.get("answer"), str) or not data["answer"].strip():
        raise ValueError("응답의 answer가 비어 있습니다.")
    return data


In [5]:
# 반환된 토큰 사용량으로 할인 없는 비용을 추정합니다.
def cost_usd(records: list[dict]) -> float:
    return (sum(int(r["input"]) for r in records) * 0.75
            + sum(int(r["output"]) for r in records) * 4.50) / 1_000_000
def score_of(data: dict) -> int:
    value = data.get("score")
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise ValueError("평가 응답의 score는 숫자여야 합니다.")
    if not 0 <= value <= 100:
        raise ValueError("평가 응답의 score는 0~100이어야 합니다.")
    return int(value)


## 오케스트레이터가 입력에 맞춰 작업 분해

In [6]:
# 입력에 필요한 하위 작업을 모델이 동적으로 만들게 합니다.
task = "전 직원에게 2단계 인증 도입을 설명하고, 등록 절차와 지원 방법을 포함한 안내문을 작성하세요."
plan = await call_json(
    "작업 오케스트레이터입니다. 입력에 필요한 하위 작업을 동적으로 설계합니다.",
    task + "\nJSON tasks 배열에 2~4개의 id, instruction을 반환하세요. instruction은 각각 40자 이내입니다.", 600)
raw_tasks = plan.get("tasks", [])
tasks = [item for item in raw_tasks if isinstance(item, dict)
         and item.get("id") and item.get("instruction")][:4]
if len(tasks) < 2:
    raise ValueError("오케스트레이터가 유효한 하위 작업을 2개 이상 만들지 못했습니다.")
print("동적 작업 수:", len(tasks))
for item in tasks:
    print(item["id"], "-", item["instruction"])


동적 작업 수: 3
task1 - 2단계 인증 도입 목적을 설명
task2 - 등록 절차를 간단히 안내
task3 - 지원 및 문의 방법을 정리


## 계획에 생긴 작업 수만큼 워커 실행

In [7]:
# 계획에 포함된 작업 수만큼 워커를 동시에 실행합니다.
async def run_worker(item: dict) -> dict:
    result = await call_answer(
        "독립 워커입니다. 맡은 범위만 조사하거나 작성합니다.",
        f"전체 과제: {task}\n담당: {item['instruction']}\nJSON result를 120자 이내로 반환하세요.", 400)
    return {"id": item["id"], "result": result["answer"]}
worker_results = await asyncio.gather(*(run_worker(item) for item in tasks))
print("실행한 워커:", [item["id"] for item in worker_results])
print("고정 목록 사용:", False)


실행한 워커: ['task1', 'task2', 'task3']
고정 목록 사용: False


## 오케스트레이터가 워커 결과를 종합

In [8]:
# 검사된 워커 결과를 엄격한 answer 형식으로 종합합니다.
synthesis = await call_answer(
    "편집 오케스트레이터입니다. 중복을 줄이고 사실을 보존합니다.",
    f"과제: {task}\n워커 결과: {worker_results}\nJSON answer 하나만 300자 이내로 작성하세요.", 700)
final_answer = synthesis["answer"]
print("최종 결과:", final_answer)
print("종합 응답 상태: ok")
print("호출 수:", len(calls), "= 계획 1 + 워커", len(tasks), "+ 종합 1")
print("사용량 기준 추정 비용($):", round(cost_usd(calls), 6))
assert final_answer


최종 결과: 전 직원 대상 2단계 인증을 도입합니다. 비밀번호가 유출·도용되더라도 계정과 회사 데이터를 추가로 보호해 보안을 강화합니다. 등록 절차: ① 안내 메일의 등록 링크 클릭 ② 회사 계정 로그인 ③ 휴대폰 앱 또는 문자 인증 방식 선택 ④ 화면 안내에 따라 설정 완료. 지원이 필요하면 IT헬프데스크(내선 1234, it-support@company.com)로 문의해 주세요. 문의 시 사번, 오류 화면, 발생 시간을 함께 보내주시면 빠르게 도와드리겠습니다.
종합 응답 상태: ok
호출 수: 5 = 계획 1 + 워커 3 + 종합 1
사용량 기준 추정 비용($): 0.002071


## 실험 결과 해석

작업 목록은 코드에 고정하지 않았습니다. 같은 입력에서도 모델이 2~4개 범위에서 다른 분해를 만들 수 있습니다. 유효한 작업이 2개보다 적으면 조용히 기본 목록으로 바꾸지 않고 오류로 중단합니다. 그래야 오케스트레이터의 실제 실패를 관찰할 수 있습니다.